# DisasterIQ — Kaggle GPU fine-tune

Fine-tune PyTorch xView2 on your pre-built **train_subset** (~1449 pairs).

**Before running:**
1. Upload `disasteriq-train-subset.zip` as a private Kaggle dataset (see `docs/KAGGLE_FINETUNE.md`)
2. **Input** → add dataset `disasteriq-train-subset`
3. **Settings** → **Accelerator** → **GPU** (T4 or P100)

**Run order:** Cells 1–5 (CPU ok) → Cell 6 (must show CUDA True) → Cell 7 (training) → Cell 8 (verify output)

In [ ]:
# Edit if your dataset slug differs
DATASET_SLUG = "disasteriq-train-subset"
REPO_URL = "https://github.com/AhmadRaza4076/DisasterIQ.git"
REPO_BRANCH = "main"
XVIEW2_URL = "https://github.com/michal2409/xView2.git"

from pathlib import Path

WORKING = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
REPO_ROOT = WORKING / "DisasterIQ"

candidates = [
    INPUT / DATASET_SLUG,
    INPUT / DATASET_SLUG / "train_subset",
]
dataset_path = next((p for p in candidates if (p / "images").is_dir()), None)
if dataset_path is None:
    print("Available input dirs:", [p.name for p in INPUT.iterdir()] if INPUT.exists() else [])
    raise FileNotFoundError(
        f"Dataset not found. Add Input dataset '{DATASET_SLUG}' with images/ labels/ targets/"
    )
print("Dataset OK:", dataset_path)
print("Images:", len(list((dataset_path / "images").glob("*"))))

In [ ]:
!pip install -q albumentations opencv-python-headless pyyaml joblib tqdm timm segmentation-models-pytorch

In [ ]:
import os
import shutil
from pathlib import Path

if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)
os.chdir(WORKING)
!git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} DisasterIQ

xview2_dir = REPO_ROOT / "ml" / "pytorch-xview2"
if xview2_dir.exists():
    shutil.rmtree(xview2_dir)
!git clone --depth 1 {XVIEW2_URL} {xview2_dir}
print("Repos ready at", REPO_ROOT)

In [ ]:
import os
os.chdir(REPO_ROOT)
!chmod +x ml/finetune/run_kaggle_pipeline.sh ml/finetune/train_localization.sh ml/finetune/train_damage.sh
!bash ml/finetune/run_kaggle_pipeline.sh --prep-only

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    raise RuntimeError(
        "No GPU. Kaggle: Settings → Accelerator → GPU → Save, wait for restart, re-run this cell."
    )

In [ ]:
import os
os.chdir(REPO_ROOT)
!bash ml/finetune/run_kaggle_pipeline.sh --train-only

In [ ]:
from pathlib import Path

ckpt = Path("/kaggle/working/damage_best.ckpt")
alt = Path("/kaggle/working/results/dmg/checkpoints/best.ckpt")
if not ckpt.exists() and alt.exists():
    import shutil
    shutil.copy(alt, ckpt)

if ckpt.exists():
    print(f"SUCCESS: {ckpt} ({ckpt.stat().st_size / 1e6:.1f} MB)")
    print("Download from Output tab → damage_best.ckpt")
    print("Laptop: copy to ml/checkpoints/damage_best.ckpt")
    print(".env: INFERENCE_MODE=pytorch")
else:
    raise FileNotFoundError("No checkpoint — check training logs above")